In [4]:
import torch 
import sys
from datasets.graph_datasets.graph_heat_dataset import HeatGraphDataset
import yaml
from models.forecasting.GNO import GNO
from torch_geometric.loader import DataLoader
sys.path.append('...')

In [6]:
# Load configs
with open("../checkpoints/gno_heat/20260912_2148/model_configs.yaml", "r") as file:
    cfg = yaml.safe_load(file)

# Load in sample to run inference on

In [13]:
data_path ='../data/test_data/heat_equation_m64_h0_minmax_N200.pt'
input_data = torch.load(data_path)
simulation_idx = 0
simulation_frame = 0

X, pde_param = input_data['X'][simulation_idx,simulation_frame], input_data['a'][simulation_idx]

H, W = X.shape[-1], X.shape[-2] 
x_indices = torch.tensor([x for x in range(W)])
y_indices = torch.tensor([x for x in range(H)])
node_grid_indices = torch.cartesian_prod(x_indices, y_indices) # Collection of (x, y) grid indices
node_spatial_pos = torch.cartesian_prod(x_indices / W, y_indices / H) # Collection of (x, y) spatial positions

dataset = HeatGraphDataset('../data/test_data/heat_equation_m64_h0_minmax_N200.pt',
                           list(input_data.keys())[1:],
                           r = cfg['radius'],
                           bc ='periodic',
                           sub_graph_size = cfg['sub_graph_size'])

In [14]:
node_grid_indices

tensor([[ 0,  0],
        [ 0,  1],
        [ 0,  2],
        ...,
        [63, 61],
        [63, 62],
        [63, 63]])

# Load Model

In [5]:
gno_model = GNO(optimiser = cfg['optimiser'], 
                 learning_rate = cfg['learning_rate'], 
                 num_node_input_features = cfg['num_node_input_features'],
                 num_edge_features = cfg['num_edge_features'], 
                 num_latent_dim = cfg['num_latent_dim'], 
                 output_dim = cfg['output_dim'],
                 num_gno_layers = cfg['num_gno_layers'],
                 kernel_ffn_layers = cfg['kernel_ffn_layers'],
                 kernel_ffn_dropout = cfg['kernel_ffn_dropout'], 
                 GNO_layer_activation = cfg['gno_layer_activation'])

model_path = '../checkpoints/gno_heat/20260912_2148/gno-epoch=0009-val_loss=0.0000.ckpt'
gno_model.load_state_dict(torch.load(model_path)['state_dict'])

<All keys matched successfully>

In [9]:
import torch

A = torch.randint(low = 0, high=5, size = (5,5))

In [14]:
A

tensor([[0, 1, 1, 0, 4],
        [4, 2, 3, 4, 0],
        [2, 4, 3, 4, 1],
        [3, 2, 2, 4, 1],
        [4, 0, 4, 2, 1]])

In [16]:
A[0:3, 1:4]

tensor([[1, 1, 0],
        [2, 3, 4],
        [4, 3, 4]])

In [21]:
A.flatten()

tensor([0, 1, 1, 0, 4, 4, 2, 3, 4, 0, 2, 4, 3, 4, 1, 3, 2, 2, 4, 1, 4, 0, 4, 2,
        1])

In [ ]:
A.flatten()[1:4:5]

SyntaxError: invalid syntax (2845815802.py, line 1)